# Mohammad Amin Kiani - 4043644008
# NLP - HW3 : Recurrent To Attention
# ui.ac.ir 404-405

In [1]:
# # نصب کتابخانه‌های مورد نیاز برای پردازش مدل‌های زبانی، کوانتیزه‌سازی و تنظیم دقیق
!pip install -q -U transformers accelerate bitsandbytes peft datasets trl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.2 MB/s eta 0:00:00


#### 1:

In [5]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# نام مدل سبک که انتخاب کردیم
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# تنظیمات پرامپت‌ها برای سه وظیفه
tasks = {
    "Translation": "متن زیر را به انگلیسی ترجمه کن:\nدیروز به کتابخانه رفتم تا یک کتاب جدید برای مطالعه پیدا کنم.",
    "Summarization": "متن زیر را در یک جمله خلاصه کن:\nهوش مصنوعی در سال‌های اخیر پیشرفت‌های چشمگیری داشته است. الگوریتم‌های یادگیری عمیق توانسته‌اند در پردازش تصویر و زبان طبیعی به سطح انسان نزدیک شوند و این موضوع باعث تحول در صنایع مختلف از جمله پزشکی و خودروسازی شده است.",
    "Sentiment": "احساس متن زیر را مشخص کن (فقط یک کلمه بنویس: مثبت، منفی یا خنثی):\nمحصولی که به دستم رسید کیفیت فوق‌العاده‌ای داشت و بسته‌بندی آن بسیار زیبا بود."
}

# تابعی برای تعریف تنظیمات کوانتیزه‌سازی بر اساس نام حالت
def get_quantization_config(precision):
    if precision == "FP16":
        return None # کوانتیزه نمی‌شود، با فرمت ۱۶ بیت می‌آید
    elif precision == "INT8":
        return BitsAndBytesConfig(load_in_8bit=True)
    elif precision == "NF4":
        return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    elif precision == "INT4":
        return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="fp4", bnb_4bit_compute_dtype=torch.float16)

# تابعی برای ارزیابی مدل
def evaluate_model(precision):
    print(f"\n{'='*40}\nEvaluating precision: {precision}\n{'='*40}")

    # ریست کردن حافظه GPU برای اندازه‌گیری دقیق‌تر
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    quant_config = get_quantization_config(precision)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # بارگذاری مدل با کانفیگ مربوطه
    if precision == "FP16":
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, device_map="auto")

    # محاسبه حداکثر حافظه مصرفی (VRAM)
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3) # تبدیل بایت به گیگابایت
    print(f"Peak VRAM Usage: {peak_vram:.2f} GB")

    # اجرای وظایف
    for task_name, prompt in tasks.items():
        # تبدیل پرامپت به فرمت چت استاندارد مدل
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to(model.device)

        start_time = time.time()

        # تولید متن با تنظیمات خواسته‌شده توسط تمرین
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0,
            do_sample=False
        )

        end_time = time.time()
        latency = end_time - start_time

        # محاسبه تعداد توکن‌های تولید شده (فقط توکن‌های جدید، بدون پرامپت)
        generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        num_tokens = len(generated_tokens)
        tokens_per_sec = num_tokens / latency

        # استخراج خروجی متنی
        response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        print(f"\n--- Task: {task_name} ---")
        print(f"Latency: {latency:.2f} sec | Tokens/sec: {tokens_per_sec:.2f}")
        print(f"Output: {response.strip()}")

    # پاک کردن مدل از حافظه برای دور بعدی
    del model
    del tokenizer
    torch.cuda.empty_cache()

# اجرای حلقه روی تمام حالت‌ها
precisions = ["FP16", "INT8", "NF4", "INT4"]
for p in precisions:
    evaluate_model(p)


Evaluating precision: FP16


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Peak VRAM Usage: 3.30 GB

--- Task: Translation ---
Latency: 0.68 sec | Tokens/sec: 22.16
Output: Doris wrote to the library to find a new book for research.

--- Task: Summarization ---
Latency: 5.70 sec | Tokens/sec: 22.46
Output: در سال‌های اخیر، هوش مصنوعی پیشرفت‌های چشمگیری داشت و الگوریتم‌های یادگیری عمیق همراه بودند. این موضوع باعث تحول در صنایع مختلف، از پزشکی به خودروسازی، به منظور کاهش آثار مشکلات و توسعه فناوری‌ها، به روزرسانی و تحسین عملکرد و کاربردهای

--- Task: Sentiment ---
Latency: 0.17 sec | Tokens/sec: 23.84
Output: منفی

Evaluating precision: INT8


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Peak VRAM Usage: 2.98 GB


...


--- Task: Sentiment ---
Latency: 1.89 sec | Tokens/sec: 2.12
Output: منفی

Evaluating precision: NF4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Peak VRAM Usage: 2.83 GB

--- Task: Translation ---
Latency: 2.41 sec | Tokens/sec: 10.35
Output: Here is the translation of that sentence into English:

I sent Droid to search for a new book to study.
)

--- Task: Summarization ---
Latency: 11.89 sec | Tokens/sec: 10.76
Output: در اینجا یک جمله برای یافتن یک زمانی زیادی برای یافتن یک یادگیری گذاری یا گروه گذاری یا یک گروه گفاهای یا گروه گفاهای یا گروه گفاهای یا گروه گفاهای یا گروه گفاهای یا گروه

--- Task: Sentiment ---
Latency: 0.40 sec | Tokens/sec: 10.00
Output: منفی

Evaluating precision: INT4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Peak VRAM Usage: 3.02 GB

--- Task: Translation ---
Latency: 6.38 sec | Tokens/sec: 10.50
Output: Here's the English translation of the given text:

I am looking for a book that I can read to learn about the world.

This is an English version of the original text in simple English. The original text was in Persian and it translates to "I am searching for a book that I can read to learn about the world."

--- Task: Summarization ---
Latency: 11.43 sec | Tokens/sec: 11.20
Output: برخه، معمولاً: "هوش مصنوعی در سال‌های اخیر پیشرفت‌های چشمگیری داشته است. الگوریتم‌های یادگیری عمیق توانسته‌اند در پردازش تصویر و زبان طبیعی به سطح انسان نزدیک شوند و این موضوع باعث تحول در صنایع مختلف از جمله پزشکی و خودرو

--- Task: Sentiment ---
Latency: 0.37 sec | Tokens/sec: 10.68
Output: منفی
